In [1]:
import numpy as np
import pandas as pd

# 독성 네트워크 구축

In [2]:
DILI = pd.read_csv('data/DILI.csv')
OFT = pd.read_csv('data/OpenFoodTox_non_toxic_salt_filtered.csv')
T3DB = pd.read_csv('data/T3DB_food_hepototixic_salt_filtered.csv')
US = pd.read_csv('data/useful_substances.csv')

# 각 DataFrame에 origin 컬럼 추가
DILI = DILI.assign(origin='DILI')
OFT = OFT.assign(origin='OFT')
T3DB = T3DB.assign(origin='T3DB')
US = US.assign(origin='US')

print(f'length of DILI: {len(DILI)}, '
      f'length of OpenFoodTox: {len(OFT)}, length of T3DB: {len(T3DB)}, '
      f'length of useful substances: {len(US)}')

length of DILI: 475, length of OpenFoodTox: 123, length of T3DB: 21, length of useful substances: 16


In [3]:
TOXIC_TOTAL = pd.read_csv('data/TOXIC_TOTAL.csv')
TOXIC_TOTAL

,Drug_ID,Drug,Y,origin,CAN_SMILES,Substance_Name,Toxicity_Flag,canonical_smiles,DB_Chemical_name,Hepatotoxicity,Number,SMILES,ID
0,187.0,CC(=O)OCC[N+](C)(C)C,0.0,DILI,CC(=O)OCC[N+](C)(C)C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0_DILI
1,247.0,C[N+](C)(C)CC(=O)[O-],0.0,DILI,C[N+](C)(C)CC(=O)[O-],NaN,NaN,NaN,NaN,NaN,NaN,NaN,1_DILI
2,298.0,O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl,0.0,DILI,O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2_DILI
3,338.0,O=C(O)c1ccccc1O,0.0,DILI,O=C(O)c1ccccc1O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3_DILI
4,444.0,CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1,0.0,DILI,CC(NC(C)(C)C)C(=O)c1cccc(Cl)c1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4_DILI
...,...,...,...,...,...,...,...,...,...,...,...,...,...
568,NaN,NaN,NaN,US,O=c1c(-c2ccc(O)cc2)coc2cc(O)cc(O)c12,NaN,NaN,NaN,NaN,NaN,Genistein,O=c1c(-c2ccc(O)cc2)coc2cc(O)cc(O)c12,568_US
569,NaN,NaN,NaN,US,CC(=O)OC1COC(OC2C(O)COC(OC3C(O)C(C)(C)CC4C5=CC...,NaN,NaN,NaN,NaN,NaN,aa,CC(=O)OC1COC(OC2C(O)COC(OC3C(O)C(C)(C)CC4C5=CC...,569_US
570,NaN,NaN,NaN,US,CC(=O)OC[C@H]1O[C@@H](O[C@@H]2[C@@H](O)[C@H](O...,NaN,NaN,NaN,NaN,NaN,ab,CC(=O)OC[C@H]1O[C@@H](O[C@@H]2[C@@H](O)[C@H](O...,570_US
571,NaN,NaN,NaN,US,CC1(C)C[C@@H](O)[C@]2(C)CC[C@]3(C)C(=CC[C@@H]4...,NaN,NaN,NaN,NaN,NaN,ba,CC1(C)C[C@@H](O)[C@]2(C)CC[C@]3(C)C(=CC[C@@H]4...,571_US


In [4]:
# TOXIC_TOTAL: 기존 DataFrame, 반드시 'ID', 'Y' 두 컬럼이 존재해야 합니다.
# 1) 각 규칙에 대한 Boolean 조건 벡터를 만듭니다.
cond_dili_0 = TOXIC_TOTAL['ID'].str.endswith('_DILI') & (TOXIC_TOTAL['Y'] == 0)
cond_dili_1 = TOXIC_TOTAL['ID'].str.endswith('_DILI') & (TOXIC_TOTAL['Y'] == 1)
cond_oft    = TOXIC_TOTAL['ID'].str.endswith('_OFT')
cond_t3db   = TOXIC_TOTAL['ID'].str.endswith('_T3DB')
cond_us     = TOXIC_TOTAL['ID'].str.endswith('_US')

# 2) 조건 순서와 결과 값을 나란히 두고 np.select로 한꺼번에 라벨링
conditions = [cond_dili_0, cond_dili_1, cond_oft, cond_t3db, cond_us]
choices    = [0,           1,           0,        1,         0]

TOXIC_TOTAL['label'] = np.select(conditions, choices, default=np.nan)
TOXIC_TOTAL['label'].value_counts()
# TOXIC_TOTAL['label'].astype(str)
# TOXIC_TOTAL.reset_index(drop=True, inplace=True)

TOXIC_TOTAL = TOXIC_TOTAL.drop(columns=['SMILES'])
TOXIC_TOTAL = TOXIC_TOTAL.rename(columns={'CAN_SMILES': 'SMILES'})

TOXIC_TOTAL.reset_index(drop=True, inplace=True)
TOXIC_TOTAL.to_csv('data/TOXIC_TOTAL_labeled.csv', index=False)


In [5]:
TOXIC_TOTAL['label'].value_counts()

0.0    317
1.0    256
Name: label, dtype: int64

# SSM

In [ ]:
# murcho scaffold split
pubchemmerge_clean = pd.read_csv('data/TOXIC_TOTAL_labeled.csv')
# pubchemmerge_clean
from collections import defaultdict

import numpy
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from chainer_chemistry.dataset.splitters.base_splitter import BaseSplitter


def generate_scaffold(smiles, include_chirality=False):
    """return scaffold string of target molecule"""
    mol = Chem.MolFromSmiles(smiles)
    scaffold = MurckoScaffold\
        .MurckoScaffoldSmiles(mol=mol, includeChirality=include_chirality)
    return scaffold


class ScaffoldSplitter(BaseSplitter):
    """Class for doing data splits by chemical scaffold.

    Referred Deepchem for the implementation, https://git.io/fXzF4
    """
    def _split(self, dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1,
               **kwargs):
        numpy.testing.assert_almost_equal(frac_train + frac_valid + frac_test,
                                          1.)
        seed = kwargs.get('seed', None)
        smiles_list = kwargs.get('smiles_list')
        include_chirality = kwargs.get('include_chirality')
        if len(dataset) != len(smiles_list):
            raise ValueError("The lengths of dataset and smiles_list are "
                             "different")

        rng = numpy.random.RandomState(seed)

        scaffolds = defaultdict(list) #! scaffold-compound dictionary
        for ind, smiles in enumerate(smiles_list):
            scaffold = generate_scaffold(smiles, include_chirality)
            scaffolds[scaffold].append(ind)

        # scaffold_sets = rng.permutation(list(scaffolds.values()))

        # n_total_valid = int(numpy.floor(frac_valid * len(dataset)))
        # n_total_test = int(numpy.floor(frac_test * len(dataset)))
        
        scaffold_sets = list(scaffolds.values())
        rng.shuffle(scaffold_sets)

        train_index = []
        valid_index = []
        test_index = []

        for scaffold_set in scaffold_sets:
            if len(valid_index) < frac_valid * len(dataset):
            # if len(valid_index) + len(scaffold_set) <= n_total_valid:
                valid_index.extend(scaffold_set)
            elif len(test_index) < frac_test * len(dataset):
            # elif len(test_index) + len(scaffold_set) <= n_total_test:
                test_index.extend(scaffold_set)
            else:
                train_index.extend(scaffold_set)

        return numpy.array(train_index), numpy.array(valid_index),\
            numpy.array(test_index),\


    def train_valid_test_split(self, dataset, smiles_list, frac_train=0.8,
                               frac_valid=0.1, frac_test=0.1, converter=None,
                               return_index=True, seed=None,
                               include_chirality=False, **kwargs):
        """Split dataset into train, valid and test set.

        Split indices are generated by splitting based on the scaffold of small
        molecules.

        Args:
            dataset(NumpyTupleDataset, numpy.ndarray):
                Dataset.
            smiles_list(list):
                SMILES list corresponding to datset.
            seed (int):
                Random seed.
            frac_train(float):
                Fraction of dataset put into training data.
            frac_valid(float):
                Fraction of dataset put into validation data.
            converter(callable):
            return_index(bool):
                If `True`, this function returns only indices. If `False`, this
                function returns splitted dataset.

        Returns:
            SplittedDataset(tuple): splitted dataset or indices

        """
        return super(ScaffoldSplitter, self)\
            .train_valid_test_split(dataset, frac_train, frac_valid, frac_test,
                                    converter, return_index, seed=seed,
                                    smiles_list=smiles_list,
                                    include_chirality=include_chirality,
                                    **kwargs)

    def train_valid_split(self, dataset, smiles_list, frac_train=0.9,
                          frac_valid=0.1, converter=None, return_index=True,
                          seed=None, include_chirality=False, **kwargs):
        """Split dataset into train and valid set.

        Split indices are generated by splitting based on the scaffold of small
        molecules.

        Args:
            dataset(NumpyTupleDataset, numpy.ndarray):
                Dataset.
            smiles_list(list):
                SMILES list corresponding to datset.
            seed (int):
                Random seed.
            frac_train(float):
                Fraction of dataset put into training data.
            frac_valid(float):
                Fraction of dataset put into validation data.
            converter(callable):
            return_index(bool):
                If `True`, this function returns only indices. If `False`, this
                function returns splitted dataset.

        Returns:
            SplittedDataset(tuple): splitted dataset or indices

        """
        return super(ScaffoldSplitter, self)\
            .train_valid_split(dataset, frac_train, frac_valid, converter,
                               return_index, seed=seed,
                               smiles_list=smiles_list,
                               include_chirality=include_chirality, **kwargs)

# from chainer_chemistry.dataset.splitters.scaffold_splitter import ScaffoldSplitter

smiles_list = pubchemmerge_clean['SMILES'].tolist()
dataset = np.arange(len(pubchemmerge_clean))

splitter = ScaffoldSplitter()

# Perform the split with frac_valid set to 0
train_indices, valid_indices, test_indices = splitter.train_valid_test_split(
    pubchemmerge_clean,
    smiles_list= smiles_list,
    frac_train=0.8,
    frac_valid=0.0,
    frac_test=0.2,
    seed=42
)

# Print the split indices
print("Train indices:", train_indices)
print("Valid indices:", valid_indices)
print("Test indices:", test_indices)


len(train_indices)
len(test_indices)
0 in train_indices
pubchemmerge_clean_sample_test = pubchemmerge_clean.loc[test_indices]
pubchemmerge_clean_sample_test['label'].value_counts()
import numpy as np
train_indices.max()
# valid_indices.max()
test_indices.max()

# Check for overlapping indices
overlap_indices = np.intersect1d(train_indices, np.intersect1d(valid_indices, test_indices))

if len(overlap_indices) > 0:
    print("There are overlapping indices between train_indices, valid_indices, and test_indices.")
    print("Overlapping indices:", overlap_indices)
else:
    print("There are no overlapping indices between train_indices, valid_indices, and test_indices.")

import pandas as pd

# pubchemmerge_clean 데이터프레임을 사용하는 가정
# train_indices, valid_indices, test_indices를 이미 구했음

# 인덱스를 사용해 train 및 test 데이터프레임을 생성
train_df = pubchemmerge_clean.iloc[train_indices]
test_df = pubchemmerge_clean.iloc[test_indices]

# train_df
# # CSV 파일로 저장
train_df.to_csv('bean/source_SUBDYVE/SSM/data/TOXIC/train_data.csv', index=False)
test_df.to_csv('bean/source_SUBDYVE/SSM/data/TOXIC/test_data.csv', index=False)

test_df = test_df[['SMILES', 'label', 'CID', 'CompoundName']]
test_df.reset_index(drop=True, inplace=True)
test_df.index.name = 'index'
test_df.to_csv('bean/source_SUBDYVE/SSM/data/TOXIC/test_data.csv')


ModuleNotFoundError: No module named 'distutils.msvccompiler'

In [7]:
#! 위의 코드 에러나서 BHI4에서 위 작업만 진행

In [36]:
train_data = pd.read_csv('./source_SUBDYVE/SSM/data/TOXIC/train_data.csv')
train_data = train_data.drop(columns=['SMILES'])
train_data = train_data.rename(columns={'CAN_SMILES': 'SMILES'})
train_data.to_csv('./source_SUBDYVE/SSM/data/TOXIC/train_data.csv', index=False)


test_data = pd.read_csv('./source_SUBDYVE/SSM/data/TOXIC/test_data.csv')
test_data = test_data.drop(columns=['SMILES'])
test_data = test_data.rename(columns={'CAN_SMILES': 'SMILES'})
test_data.to_csv('./source_SUBDYVE/SSM/data/TOXIC/test_data.csv', index=False)


# Seed 파일 & Comp 파일 만들기

In [ ]:
TOXIC_TOTAL_labeled = pd.read_csv('data/TOXIC_TOTAL_labeled.csv')

SEED_FOR_SUBDYVE = TOXIC_TOTAL_labeled[(TOXIC_TOTAL_labeled['Y'] == 1) & (TOXIC_TOTAL_labeled['origin'] == 'DILI') | (TOXIC_TOTAL_labeled['origin'] == 'T3DB')]
SEED_FOR_SUBDYVE.to_csv('data/TOXIC_SUBDYVE_SEED.csv', index=False)

COMP_FOR_SUBDYVE = TOXIC_TOTAL_labeled[~TOXIC_TOTAL_labeled['ID'].isin(SEED_FOR_SUBDYVE['ID'])]
COMP_FOR_SUBDYVE.to_csv('data/TOXIC_SUBDYVE_COMP.csv', index=False)

# 한번만 수행
전처리 + ID 매칭 (매우 중요)

In [ ]:
SEED_FOR_SUBDYVE = pd.read_csv('data/TOXIC_SUBDYVE_SEED.csv')
COMP_FOR_SUBDYVE = pd.read_csv('data/TOXIC_SUBDYVE_COMP.csv')

In [25]:
from rdkit import Chem
import pickle
import os
import pandas as pd

# TARGETS = [
#     "ACES",
#     "ADA",
#     "ANDR",
#     "EGFR",
#     "FA10",
#     "KIT",
#     "PLK1",
#     "SRC",
#     "THRB",
#     "UROK"
# ]

# for target in TARGETS:

output_dir = f"source_SUBDYVE/SUBMINE/data/TOXIC/"
os.makedirs(output_dir, exist_ok=True)


#### ACTIVE DATA ####
SEED_DATA = pd.read_csv(f'data/SUBDYVE_SEED.csv')
# SEED_DATA.rename(columns={0: 'origin_SMILES', 1:'ID'}, inplace=True)
print(f"[INFO] SEED_DATA: {len(SEED_DATA)}")
# SEED_DATA.drop_duplicates(subset='origin_SMILES', keep='last', inplace=True)
# SEED_DATA.drop_duplicates(subset='unknown', keep='last', inplace=True)
# SEED_DATA.drop_duplicates(subset='ID', keep='last', inplace=True)
# SEED_DATA['SMILES'] = SEED_DATA['origin_SMILES'].apply(lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x)))
# SEED_DATA.drop_duplicates(subset='SMILES', keep='last', inplace=True)
# SEED_DATA = SEED_DATA.sort_values(by='SMILES', ascending=False).reset_index(drop=True)

# print(f"[INFO] 전처리 후 SEED_DATA: {len(SEED_DATA)}")
print(f"[INFO] SEED_DATA ID nunique : {SEED_DATA['ID'].nunique()}")

SEED_ID2I = {SEED_id: i for i, SEED_id in enumerate(SEED_DATA['ID'])}
pickle.dump(SEED_ID2I, open(f"{output_dir}SEED_ID2I.pkl", "wb"))


#### DECOY DATA ####
COMP_DATA = pd.read_csv(f'data/SUBDYVE_COMP.csv')
# COMP_DATA.rename(columns={0: 'origin_SMILES', 1:"unknown"}, inplace=True)

print(f"[INFO] COMP_DATA: {len(COMP_DATA)}")
# COMP_DATA.drop_duplicates(subset=['origin_SMILES'], keep='last', inplace=True)
# COMP_DATA.drop_duplicates(subset=['unknown'], keep='last', inplace=True)
# COMP_DATA['SMILES'] = COMP_DATA['origin_SMILES'].apply(lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x)))
# COMP_DATA.drop_duplicates(subset='SMILES', keep='last', inplace=True)
# COMP_DATA = COMP_DATA.sort_values(by='SMILES', ascending=False).reset_index(drop=True)
# COMP_DATA['DECOYID'] = COMP_DATA['unknown'].apply(lambda x: f"D_{x}")
# print(f"[INFO] 전처리 후 COMP_DATA: {len(COMP_DATA)}")
print(f"[INFO] COMP_DATA DECOYID nunique : {COMP_DATA['ID'].nunique()}")

COMP_ID2I = {COMP_id: i for i, COMP_id in enumerate(COMP_DATA['ID'])}
pickle.dump(COMP_ID2I, open(f"{output_dir}COMP_ID2I.pkl", "wb"))


# #### SEED DATA ####
# # if target == 'ADA':
# #     THRESHOLD = 1
# #     SEED_DATA = pd.read_csv(f'data/DUDE/{target}/pubchem_results/{target}_DATA_{THRESHOLD}.csv')
# # else:
# #     THRESHOLD = 0.9
# #     SEED_DATA = pd.read_csv(f'data/DUDE/{target}/pubchem_results/{target}_DATA_{THRESHOLD}.csv')
# # THRESHOLD = 0.9
# # SEED_DATA = pd.read_csv(f'data/DUDE/{target}/pubchem_results/{target}_DATA_{THRESHOLD}.csv')

# print(f"[INFO] SEED_DATA: {len(SEED_DATA)}")
# SEED_DATA = SEED_DATA[SEED_DATA['label'] == 1]
# SEED_DATA = SEED_DATA.sort_values(by='SMILES', ascending=False)
# SEED_DATA.drop_duplicates(subset=['SMILES'], keep='last', inplace=True)
# SEED_DATA.drop_duplicates(subset=['cid'], keep='last', inplace=True)
# SEED_DATA = SEED_DATA.sort_values(by='SMILES', ascending=False).reset_index(drop=True)
# print(f"[INFO] 전처리 후 SEED_DATA: {len(SEED_DATA)}")
# print(f"[INFO] SEED_DATA cid nunique : {SEED_DATA['cid'].nunique()}")

# SEED_ID2I = {seed_id: i for i, seed_id in enumerate(SEED_DATA['cid'])}
# pickle.dump(SEED_ID2I, open(f"{output_dir}SEED_ID2I.pkl", "wb"))

SEED_DATA.to_csv(f"{output_dir}SEED_DATA.csv", index=False)
COMP_DATA.to_csv(f"{output_dir}COMP_DATA.csv", index=False)

# print(f"✅ {target} data saved.")


[INFO] SEED_DATA: 256
[INFO] SEED_DATA ID nunique : 256
[INFO] COMP_DATA: 317
[INFO] COMP_DATA DECOYID nunique : 317


# Embedding 추출
BHI4에서 진행